# R2-A3-S12 — Mecanismos de Atención y Transformers aplicados a Series de Tiempo

**Especialización en Inteligencia Artificial — Universidad de Cundinamarca**
**Asignatura:** 604013 Deep Learning – Conceptos
**Estudiante:** Jerson Baquero Cháves
**Fecha:** _(completar)_
**Repositorio GitHub:** _(pegar enlace)_

---

## Objetivo

Implementar desde cero un mecanismo de atención y una arquitectura Transformer básica, aplicados al
pronóstico de una serie de tiempo univariada, evaluando su desempeño frente a una arquitectura
recurrente (LSTM) y a un baseline estadístico ingenuo.

## Contenido

0. Configuración del entorno
1. Construcción del conjunto de datos
2. Preparación y preprocesamiento
3. Mecanismo de atención desde cero
4. Arquitectura Transformer
5. Entrenamiento
6. Modelos de referencia (LSTM y baseline ingenuo)
7. Evaluación comparativa
8. Interpretabilidad: análisis de los pesos de atención
9. Conclusiones

## 0. Configuración del entorno

In [ ]:
import math, time, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__, "| Dispositivo:", DEVICE)

plt.rcParams["figure.figsize"] = (11, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

## 1. Construcción del conjunto de datos

Se utiliza una serie de tiempo sintética que simula la **demanda eléctrica diaria** de una zona urbana
durante tres años (1.095 observaciones). La serie se construye de forma controlada para que contenga
las estructuras temporales que el mecanismo de atención debe ser capaz de descubrir:

| Componente | Descripción | Efecto esperado sobre la atención |
|---|---|---|
| Tendencia lineal | Crecimiento sostenido de la demanda | Dependencia del nivel reciente |
| Estacionalidad anual | Ciclo de 365,25 días (clima) | Variación lenta, capturada por el nivel |
| Estacionalidad semanal | Ciclo de 7 días | Atención sobre los rezagos 7 y 14 |
| Efecto fin de semana | Caída en sábado y domingo | Refuerza el patrón de 7 días |
| Ruido gaussiano | σ = 1,2 | Define el piso de error alcanzable |

Trabajar con una serie de generación conocida permite **verificar objetivamente** si el modelo aprendió
la estructura correcta: si la atención se concentra en los rezagos múltiplos de 7, el mecanismo está
capturando la estacionalidad semanal real y no un artefacto.

In [ ]:
def generar_serie(n_dias=1095, seed=SEED):
    rng = np.random.default_rng(seed)
    t = np.arange(n_dias)
    tendencia = 0.015 * t
    anual     = 8.0 * np.sin(2 * np.pi * t / 365.25)
    semanal   = 5.0 * np.sin(2 * np.pi * t / 7) + 2.5 * np.cos(2 * np.pi * t / 7)
    finde     = np.where(t % 7 >= 5, -6.0, 0.0)
    ruido     = rng.normal(0, 1.2, n_dias)
    return 60 + tendencia + anual + semanal + finde + ruido

serie = generar_serie()
fechas = pd.date_range("2023-01-01", periods=len(serie), freq="D")
df = pd.DataFrame({"fecha": fechas, "demanda": serie}).set_index("fecha")

print(df.describe().T)
df.head()

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 7))
axes[0].plot(df.index, df["demanda"], lw=0.8)
axes[0].set_title("Serie completa (3 años)")
axes[0].set_ylabel("Demanda (GWh)")

zoom = df.iloc[:56]
axes[1].plot(zoom.index, zoom["demanda"], marker="o", ms=3)
axes[1].set_title("Detalle: primeras 8 semanas (se observa el ciclo de 7 días)")
axes[1].set_ylabel("Demanda (GWh)")
plt.tight_layout(); plt.show()

## 2. Preparación y preprocesamiento

El problema se plantea como **pronóstico un paso adelante**: a partir de una ventana de los últimos
`L = 30` días se predice el valor del día siguiente (`H = 1`).

Dos decisiones metodológicas importantes:

1. **La partición es temporal, no aleatoria.** En series de tiempo, mezclar aleatoriamente las ventanas
   produce fuga de información (*data leakage*): el modelo vería el futuro durante el entrenamiento y
   las métricas quedarían infladas. Se usa 70 % entrenamiento, 15 % validación y 15 % prueba en orden
   cronológico.
2. **La estandarización se ajusta solo con el tramo de entrenamiento.** Calcular media y desviación
   sobre toda la serie también constituiría fuga de información.

In [ ]:
L, H = 30, 1

def crear_ventanas(x, L, H):
    X = np.stack([x[i:i+L]       for i in range(len(x) - L - H + 1)])
    y = np.stack([x[i+L:i+L+H]   for i in range(len(x) - L - H + 1)])
    return X.astype(np.float32), y.astype(np.float32)

X, y = crear_ventanas(serie, L, H)

n_tr = int(0.70 * len(X))
n_va = int(0.85 * len(X))

# Estandarizacion ajustada UNICAMENTE con el tramo de entrenamiento
mu, sigma = X[:n_tr].mean(), X[:n_tr].std()
Xs, ys = (X - mu) / sigma, (y - mu) / sigma

def to_tensor(a):
    return torch.tensor(a, dtype=torch.float32)

X_tr, y_tr = to_tensor(Xs[:n_tr]).unsqueeze(-1),      to_tensor(ys[:n_tr])
X_va, y_va = to_tensor(Xs[n_tr:n_va]).unsqueeze(-1),  to_tensor(ys[n_tr:n_va])
X_te, y_te = to_tensor(Xs[n_va:]).unsqueeze(-1),      to_tensor(ys[n_va:])

print(f"Entrenamiento: {X_tr.shape}  Validación: {X_va.shape}  Prueba: {X_te.shape}")
print(f"mu = {mu:.3f}   sigma = {sigma:.3f}")

## 3. Mecanismo de atención desde cero

El mecanismo central del Transformer es la **atención por producto punto escalado**
(*scaled dot-product attention*), definida en Vaswani et al. (2017) como:

$$\mathrm{Attention}(Q,K,V)=\mathrm{softmax}\!\left(\frac{QK^{\top}}{\sqrt{d_k}}\right)V$$

Interpretación en el contexto de una serie de tiempo, donde cada posición de la ventana es un día:

- **Q (*query*)**: lo que el día actual "pregunta" al resto de la ventana.
- **K (*key*)**: la etiqueta con la que cada día pasado se "anuncia".
- **V (*value*)**: la información que ese día aporta si resulta relevante.
- El producto $QK^{\top}$ mide la compatibilidad entre cada par de días.
- La división por $\sqrt{d_k}$ evita que, al crecer la dimensión, los productos punto tomen magnitudes
  grandes que saturen la softmax y anulen el gradiente.
- La **softmax** convierte esas compatibilidades en pesos que suman 1: una distribución de atención.

A diferencia de una red recurrente, que propaga la información paso a paso y tiende a diluir las
dependencias lejanas, la atención conecta **cualquier par de posiciones en un solo paso**, con un
camino de longitud constante. Esa es la ventaja estructural que se busca verificar empíricamente en la
sección 8.

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None, dropout=None):
    # Q, K, V: (batch, heads, longitud, d_k)
    d_k = Q.size(-1)
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float("-inf"))
    pesos = torch.softmax(scores, dim=-1)
    if dropout is not None:
        pesos = dropout(pesos)
    return torch.matmul(pesos, V), pesos

# Demostracion sobre una ventana real, sin entrenar: Q = K = V = la propia ventana proyectada
ventana = X_te[0].unsqueeze(0)                       # (1, L, 1)
proj = nn.Linear(1, 16)
torch.manual_seed(SEED)
h_demo = proj(ventana).unsqueeze(1)                  # (1, 1, L, 16)
salida, pesos_demo = scaled_dot_product_attention(h_demo, h_demo, h_demo)

print("Entrada :", ventana.shape)
print("Salida  :", salida.shape)
print("Pesos   :", pesos_demo.shape)
print("Suma por fila (debe ser 1.0):", pesos_demo.sum(-1).flatten()[:5].tolist())

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(pesos_demo[0, 0].detach().numpy(), cmap="viridis")
ax.set_title("Matriz de atención sin entrenar (30 × 30)")
ax.set_xlabel("Día atendido (key)"); ax.set_ylabel("Día que consulta (query)")
plt.colorbar(im, label="peso de atención"); plt.tight_layout(); plt.show()

> **Nota.** Esta matriz no mide todavía nada aprendido: las proyecciones están inicializadas
> al azar. Conviene observar que aun así aparecen bandas verticales con separación semanal.
> No son evidencia de aprendizaje, sino consecuencia de que aquí Q, K y V son la propia
> ventana proyectada: los días con valores parecidos se atienden entre sí y la serie ya tiene
> ciclo de siete días. Lo relevante es que **todas las filas son casi idénticas**, es decir, la
> atención no depende de qué posición pregunta. En la sección 8 se compara contra la matriz
> entrenada, donde sí aparece esa dependencia.

## 4. Arquitectura Transformer

Se implementa un **encoder** Transformer, suficiente para una tarea de regresión sobre una secuencia de
entrada (no se requiere decoder porque no se genera una secuencia de salida autoregresiva).

Componentes, en orden:

1. **Proyección de entrada**: lleva cada escalar diario a un espacio de dimensión `d_model`. Es el
   análogo del *embedding* en NLP, pero para valores continuos.
2. **Codificación posicional sinusoidal**: la atención es invariante al orden, por lo que sin esta
   codificación el modelo no distinguiría una ventana de su permutación. Se suman senos y cosenos de
   frecuencias geométricamente espaciadas, lo que permite al modelo representar desplazamientos
   relativos mediante combinaciones lineales.
3. **Atención multi-cabeza**: se ejecutan `n_heads` atenciones en subespacios distintos, de modo que
   cada cabeza puede especializarse (por ejemplo, una en el rezago 1 y otra en el rezago 7).
4. **Conexiones residuales y normalización por capa**: estabilizan el entrenamiento y preservan el
   gradiente en profundidad.
5. **Red feed-forward por posición**: dos capas densas con no linealidad, aplicadas de forma idéntica a
   cada posición; aporta la capacidad de transformación no lineal.
6. **Cabeza de predicción**: se toma la representación de la **última posición** de la ventana —el día
   más reciente, que es el que naturalmente consulta al pasado— y se proyecta al horizonte de pronóstico.

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len, dtype=torch.float).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, : x.size(1)]


class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0, "d_model debe ser divisible entre n_heads"
        self.n_heads, self.d_k = n_heads, d_model // n_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def _split(self, x):
        B, L, _ = x.shape
        return x.view(B, L, self.n_heads, self.d_k).transpose(1, 2)

    def forward(self, x):
        B, L, _ = x.shape
        Q, K, V = self._split(self.W_q(x)), self._split(self.W_k(x)), self._split(self.W_v(x))
        ctx, pesos = scaled_dot_product_attention(Q, K, V, dropout=self.dropout)
        ctx = ctx.transpose(1, 2).contiguous().view(B, L, -1)
        return self.W_o(ctx), pesos


class EncoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.attn  = MultiHeadSelfAttention(d_model, n_heads, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(d_ff, d_model),
        )
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        a, pesos = self.attn(x)
        x = self.norm1(x + self.drop(a))          # residual + norm
        x = self.norm2(x + self.drop(self.ff(x))) # residual + norm
        return x, pesos


class TimeSeriesTransformer(nn.Module):
    def __init__(self, n_features=1, d_model=64, n_heads=4, d_ff=128,
                 n_layers=2, dropout=0.1, horizon=1):
        super().__init__()
        self.input_proj = nn.Linear(n_features, d_model)
        self.pos_enc = PositionalEncoding(d_model)
        self.blocks = nn.ModuleList(
            [EncoderBlock(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)]
        )
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, horizon)

    def forward(self, x, return_attn=False):
        h = self.pos_enc(self.input_proj(x))
        attn_por_capa = []
        for blk in self.blocks:
            h, pesos = blk(h)
            attn_por_capa.append(pesos)
        h = self.norm(h)
        salida = self.head(h[:, -1])   # representación del día más reciente
        return (salida, attn_por_capa) if return_attn else salida

In [ ]:
modelo = TimeSeriesTransformer(d_model=64, n_heads=4, d_ff=128, n_layers=2, dropout=0.1).to(DEVICE)

n_par = sum(p.numel() for p in modelo.parameters() if p.requires_grad)
print(f"Parámetros entrenables: {n_par:,}")

# Verificación de dimensiones antes de entrenar
with torch.no_grad():
    out, attns = modelo(X_te[:4].to(DEVICE), return_attn=True)
print("Salida:", tuple(out.shape), "| Atención por capa:", [tuple(a.shape) for a in attns])

## 5. Entrenamiento

Se emplea `Adam` con pérdida de error cuadrático medio y **parada temprana** sobre el conjunto de
validación, conservando los pesos de la mejor época. La misma función de entrenamiento se reutiliza
para el modelo recurrente, de modo que la comparación de la sección 7 sea justa.

In [ ]:
def entrenar(modelo, X_tr, y_tr, X_va, y_va, epocas=150, lr=1e-3,
             batch=64, paciencia=20, verbose=True):
    modelo = modelo.to(DEVICE)
    opt = torch.optim.Adam(modelo.parameters(), lr=lr)
    crit = nn.MSELoss()
    ds = torch.utils.data.TensorDataset(X_tr, y_tr)
    dl = torch.utils.data.DataLoader(ds, batch_size=batch, shuffle=True)

    Xv, yv = X_va.to(DEVICE), y_va.to(DEVICE)
    hist = {"train": [], "val": []}
    mejor, mejor_estado, espera = float("inf"), None, 0
    t0 = time.time()

    for ep in range(1, epocas + 1):
        modelo.train(); acum = 0.0
        for xb, yb in dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            perdida = crit(modelo(xb), yb)
            perdida.backward()
            torch.nn.utils.clip_grad_norm_(modelo.parameters(), 1.0)
            opt.step()
            acum += perdida.item() * len(xb)
        tr = acum / len(ds)

        modelo.eval()
        with torch.no_grad():
            va = crit(modelo(Xv), yv).item()

        hist["train"].append(tr); hist["val"].append(va)

        if va < mejor - 1e-5:
            mejor, espera = va, 0
            mejor_estado = {k: v.detach().clone() for k, v in modelo.state_dict().items()}
        else:
            espera += 1
            if espera >= paciencia:
                if verbose: print(f"Parada temprana en la época {ep}")
                break
        if verbose and ep % 10 == 0:
            print(f"Época {ep:3d} | train {tr:.4f} | val {va:.4f}")

    modelo.load_state_dict(mejor_estado)
    if verbose:
        print(f"Mejor pérdida de validación: {mejor:.4f} | Tiempo: {time.time()-t0:.1f} s")
    return modelo, hist

modelo, hist = entrenar(modelo, X_tr, y_tr, X_va, y_va)

In [ ]:
plt.figure(figsize=(9, 4))
plt.plot(hist["train"], label="Entrenamiento")
plt.plot(hist["val"],   label="Validación")
plt.xlabel("Época"); plt.ylabel("MSE (escala estandarizada)")
plt.title("Curvas de pérdida — Transformer")
plt.legend(); plt.tight_layout(); plt.show()

## 6. Modelos de referencia

Para dimensionar el desempeño del Transformer se incluyen dos referencias:

- **Baseline ingenuo (rezago 7):** predice que la demanda de mañana será igual a la de hace siete días.
  Explota directamente la estacionalidad semanal y es sorprendentemente difícil de superar.
- **LSTM:** arquitectura recurrente clásica, entrenada con el mismo procedimiento, presupuesto de
  épocas y criterio de parada. Es la comparación relevante para evaluar la ventaja de la atención.

In [ ]:
class LSTMBaseline(nn.Module):
    def __init__(self, hidden=64, n_layers=1, horizon=1, dropout=0.0):
        super().__init__()
        self.lstm = nn.LSTM(1, hidden, n_layers, batch_first=True, dropout=dropout)
        self.head = nn.Linear(hidden, horizon)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.head(out[:, -1])

torch.manual_seed(SEED)
lstm = LSTMBaseline(hidden=64)
print(f"Parámetros LSTM: {sum(p.numel() for p in lstm.parameters()):,}")
lstm, hist_lstm = entrenar(lstm, X_tr, y_tr, X_va, y_va)

## 7. Evaluación comparativa

Las métricas se calculan sobre el conjunto de prueba y **en las unidades originales** de la serie, no en
la escala estandarizada, para que sean interpretables.

In [ ]:
def desestandarizar(a):
    return a * sigma + mu

def predecir(modelo, X):
    modelo.eval()
    with torch.no_grad():
        return modelo(X.to(DEVICE)).cpu().numpy()

def metricas(y_real, y_pred):
    err = y_real - y_pred
    return {
        "MAE":  float(np.mean(np.abs(err))),
        "RMSE": float(np.sqrt(np.mean(err ** 2))),
        "MAPE (%)": float(np.mean(np.abs(err / y_real)) * 100),
    }

y_real = desestandarizar(y_te.numpy()).ravel()
pred_tr = desestandarizar(predecir(modelo, X_te)).ravel()
pred_ls = desestandarizar(predecir(lstm,   X_te)).ravel()

# Baseline ingenuo: el valor observado 7 dias antes del objetivo
idx_obj = np.arange(n_va, len(X)) + L
pred_nv = serie[idx_obj - 7]

resultados = pd.DataFrame({
    "Transformer":     metricas(y_real, pred_tr),
    "LSTM":            metricas(y_real, pred_ls),
    "Ingenuo (lag-7)": metricas(y_real, pred_nv),
}).T.round(4)

print("Piso teórico de error (ruido irreducible): MAE ≈ %.3f" % (1.2 * np.sqrt(2/np.pi)))
resultados

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4.5))
k = 90
ax.plot(y_real[:k],  label="Valor real", color="black", lw=1.8)
ax.plot(pred_tr[:k], label="Transformer", ls="--")
ax.plot(pred_ls[:k], label="LSTM", ls=":")
ax.set_title(f"Conjunto de prueba — primeros {k} días")
ax.set_xlabel("Día"); ax.set_ylabel("Demanda (GWh)")
ax.legend(); plt.tight_layout(); plt.show()

residuos = y_real - pred_tr
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].hist(residuos, bins=30, edgecolor="white")
axes[0].set_title("Distribución de residuos — Transformer"); axes[0].set_xlabel("Error")
axes[1].scatter(y_real, pred_tr, s=12, alpha=0.6)
lim = [y_real.min(), y_real.max()]
axes[1].plot(lim, lim, color="red", lw=1)
axes[1].set_title("Predicho vs. real"); axes[1].set_xlabel("Real"); axes[1].set_ylabel("Predicho")
plt.tight_layout(); plt.show()

## 8. Interpretabilidad: análisis de los pesos de atención

Esta sección es la verificación central del trabajo. Si el mecanismo de atención funciona como se
espera, los pesos de la **última posición** de la ventana —el día más reciente, que es el que alimenta
la cabeza de predicción— deberían concentrarse en los rezagos informativos: el día inmediatamente
anterior (nivel) y los múltiplos de 7 (estacionalidad semanal).

In [ ]:
with torch.no_grad():
    _, attns = modelo(X_te.to(DEVICE), return_attn=True)

# attns[capa]: (batch, cabezas, L, L) -> promedio sobre el conjunto de prueba
A0 = attns[0].mean(dim=0).cpu().numpy()   # (cabezas, L, L)
A_prom = A0.mean(axis=0)                  # promedio tambien sobre cabezas

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(A_prom, cmap="viridis")
ax.set_title("Atención promedio — capa 1 (modelo entrenado)")
ax.set_xlabel("Día atendido (key)"); ax.set_ylabel("Día que consulta (query)")
plt.colorbar(im, label="peso"); plt.tight_layout(); plt.show()

In [ ]:
# Perfil de atencion de la ultima posicion, expresado por rezago
fila = A_prom[-1]                      # (L,)
lags = L - np.arange(L)                # posicion 29 -> lag 1 ; posicion 0 -> lag 30

fig, ax = plt.subplots(figsize=(11, 4))
colores = ["#c0392b" if l % 7 == 0 else "#95a5a6" for l in lags]
ax.bar(lags, fila, color=colores)
ax.set_xlabel("Rezago (días hacia atrás desde el día más reciente)")
ax.set_ylabel("Peso de atención")
ax.set_title("Perfil de atención del día más reciente (en rojo, rezagos múltiplos de 7)")
ax.invert_xaxis(); plt.tight_layout(); plt.show()

orden = np.argsort(fila)[::-1][:5]
print("Cinco rezagos más atendidos:")
for i in orden:
    print(f"  rezago {lags[i]:2d} días -> peso {fila[i]:.4f}")

In [ ]:
# Especializacion por cabeza: cada cabeza puede atender rezagos distintos
fig, axes = plt.subplots(1, A0.shape[0], figsize=(14, 3), sharey=True)
for h, ax in enumerate(axes):
    f = A0[h, -1]
    ax.bar(lags, f, color=["#c0392b" if l % 7 == 0 else "#95a5a6" for l in lags])
    ax.set_title(f"Cabeza {h+1}"); ax.set_xlabel("Rezago"); ax.invert_xaxis()
axes[0].set_ylabel("Peso")
plt.suptitle("Perfil de atención por cabeza", y=1.05)
plt.tight_layout(); plt.show()

### Guía de lectura de los resultados

Al ejecutar el notebook, contrastar las gráficas con las siguientes preguntas; las respuestas son el
insumo directo del análisis escrito del documento:

1. ¿Los pesos más altos coinciden con los rezagos 1, 7 y 14? Si es así, el modelo recuperó la
   estacionalidad semanal **sin que se le indicara explícitamente**.
2. ¿Las cabezas se especializaron en rezagos distintos, o todas aprendieron el mismo patrón?
3. ¿El Transformer supera al baseline ingenuo? ¿Por cuánto margen y en qué métrica?
4. ¿Qué tan cerca está el MAE del piso teórico de ruido? La diferencia es la estructura que el modelo
   aún no captura.
5. ¿Los residuos se distribuyen de forma aproximadamente simétrica y centrada en cero, o muestran
   sesgo sistemático?

## 9. Conclusiones

_(Redactar a partir de los resultados efectivamente obtenidos al ejecutar el notebook. Sugerencia de
estructura: hallazgo sobre el desempeño comparativo, hallazgo sobre interpretabilidad de la atención,
limitaciones observadas y una línea de trabajo futuro.)_

## Referencias

Vaswani, A., Shazeer, N., Parmar, N., Uszkoreit, J., Jones, L., Gomez, A. N., Kaiser, Ł., & Polosukhin, I. (2017). Attention is all you need. *Advances in Neural Information Processing Systems, 30*, 5998–6008.

In [ ]:
# Exportacion de metricas para incorporar al documento de entrega
resultados.to_csv("metricas_transformer.csv")
print(resultados.to_markdown())